<a href="https://colab.research.google.com/github/LivingstonTardzenyuy/Generative-AI/blob/main/LCEL_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import userdata
grokAI = userdata.get('GROK_api')
openAI = userdata.get('OPENAI_API_KEY')

In [1]:
!pip install langchain_chroma
!pip install langchain_community
!pip install langchain
!pip install langchain_openai

  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
Using cached langchain_core-1.1.0-py3-none-any.whl (473 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.1.0 which is incompatible.
  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached langchain_text_splitters-1.0.0-py3-none-any.whl.metadata (2.6 kB)
Using cached langchain_core-1.1.0-py3-none-any.whl (473 kB)
Using cached langchain_text_splitters-1.0.0-py3-none-any.whl (33 kB)
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langchain-text-splitters-0.

In [14]:
import bs4
from langchain import hub
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

In [7]:
# Loading the data.

loader = WebBaseLoader(
    web_paths = ["https://lilianweng.github.io/posts/2023-06-23-agent/"]
)
bs_kwargs = dict(
    parse_only = bs4.SoupStrainer(
        class_ = ('post-content', 'post-title', 'post-header')
    )
)

docs = loader.load()

# Splitting the data into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 200
)
splits = text_splitter.split_documents(docs)



In [11]:
# Performing our vector Embeddings and retrieving our data from our vector db
vectorstore = Chroma.from_documents(
    documents = splits,
    embedding = OpenAIEmbeddings(openai_api_key = openAI)
)
retriever = vectorstore.as_retriever()

In [19]:
# we are using the prompt giving by langchain. We could still decide and write ours as we have done in the past
prompt = hub.pull("rlm/rag-prompt")
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}  # Changed 'content' to 'context'
    | prompt
    | ChatOpenAI(openai_api_key = openAI, temperature = 0.8)
    | StrOutputParser()
)

In [20]:
# invoke our chain.
rag_chain.invoke("What is the Task Decomposition ?")

'Task decomposition is a technique where complex tasks are broken down into smaller and simpler steps. This allows for better planning and understanding of the task at hand. The process involves transforming big tasks into multiple manageable tasks for easier interpretation and execution.'

In [21]:
# invoke our chain.
rag_chain.invoke("what is zilotech ?")

'Zilotech is not mentioned in the provided context.'

In [23]:
!pip install langgraph

  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 4.1 MB/s eta 0:00:00
Using cached langchain_core-1.1.0-py3-none-any.whl (473 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.1.0 which is incompatible.
